<a href="https://colab.research.google.com/github/Chosencodes/Medical-Imaging-Projects/blob/main/Pneumonia_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaggle


In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c rsna-pneumonia-detection-challenge

In [ ]:
# !unzip rsna-pneumonia-detection-challenge.zip

In [ ]:
!pip install pydicom

In [ ]:
import numpy as np
import pandas as pd
import cv2
import pydicom
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm

In [ ]:
!rm -rf /content/rsna
!mkdir /content/rsna
!unzip -q rsna-pneumonia-detection-challenge.zip -d /content/rsna

In [ ]:
import os
print(os.listdir('/content/rsna/'))

In [ ]:
labels = pd.read_csv("/content/rsna/stage_2_train_labels.csv")

In [ ]:
labels.head(6)

In [ ]:
labels = labels.drop_duplicates("patientId")

In [ ]:
labels.head(6)

In [ ]:
ROOT_PATH = Path("/content/stage_2_train_images/")
SAVE_PATH = Path("Processed/")

In [ ]:
fig, axis = plt.subplots(3, 3, figsize=(10, 10))
c = 0

for i in range(3):
  for j in range(3):
    patient_id = labels.patientId.iloc[c]
    dcm_path = ROOT_PATH/patient_id
    dcm_path = dcm_path.with_suffix('.dcm')
    dcm = pydicom.dcmread(dcm_path).pixel_array
    label = labels.Target.iloc[c]

    axis[i][j].imshow(dcm, cmap="bone")
    axis[i][j].set_title(f"Label: {label}")
    c += 1

In [ ]:
sums = 0
sums_squared = 0

for c, patient_id in enumerate(tqdm(labels.patientId)):
  dcm_path = ROOT_PATH/patient_id
  dcm_path = dcm_path.with_suffix('.dcm')
  dcm = pydicom.dcmread(dcm_path).pixel_array / 255
  label = labels.Target.iloc[c]

  dcm_array = cv2.resize(dcm,(224,224)).astype(np.float16)

  train_or_val = "train" if c < 24000 else "val"

  current_save_path = SAVE_PATH/train_or_val/str(label)
  current_save_path.mkdir(parents=True, exist_ok=True)
  np.save(current_save_path/patient_id,dcm_array)

  normalizer = dcm_array.shape[0] * dcm_array.shape[1]
  if train_or_val == "train":
    sums += np.sum(dcm_array) / normalizer
    sums_squared += (np.power(dcm_array, 2).sum()) / normalizer

mean = sums / 24000
std = np.sqrt((sums_squared / 24000) - (mean**2))



In [ ]:
print(f"Mean: {mean}")
print(f"Std: {std}")

In [ ]:
def load_file(path):
  return np.load(path).astype(np.float32)

In [ ]:
!pip install torchmetrics pytorch-lightning

In [ ]:
import torch
import torchvision
from torchvision import transforms
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

In [ ]:
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.0853], std=[0.2340]),
    transforms.RandomAffine(degrees=(-5, 5), translate=(0, 0.05), scale=(0.9, 1.1)),
    transforms.RandomResizedCrop((224, 224), scale=(0.35, 1))
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.0853], std=[0.2340])
])

In [ ]:
train_data = torchvision.datasets.DatasetFolder("Processed/train/",loader=load_file,extensions="npy",transform=train_transform)

In [ ]:
val_data = torchvision.datasets.DatasetFolder("Processed/val/",loader=load_file,extensions="npy",transform=val_transform)

In [ ]:
train_loader = torch.utils.data.DataLoader(train_data,batch_size=64,shuffle=True,num_workers=4)

In [ ]:
val_loader = torch.utils.data.DataLoader(val_data,batch_size=64,num_workers=4,shuffle=False)

In [ ]:
np.unique(train_data.targets,return_counts=True), np.unique(val_data.targets,return_counts=True)

In [ ]:
torchvision.models.resnet18()

In [ ]:
class PneumoniaModule(pl.LightningModule):
  def __ini__(self,weight=1):
    super().__init__()
    self.module = torchvision.models.resnet18()
    self.model.conv1 = torch.nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    self.model.fc = torch.nn.Linear(in_features=512, out_features=1, bias=True)

    self.optimizer = torch.nn.optim.Adam(self.models.parameters(),lr=1e-4)
    self.loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([weight]))

    self.train_acc = torchmetrics.Accuracy()
    self.val_acc = torchmetrics.Accuracy()